In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/raw/ai4i2020.csv')

# Recreate temp_diff (from EDA findings)
df['temp_diff'] = (df['Process temperature [K]']
                   - df['Air temperature [K]'])

print("Loaded:", df.shape)

Loaded: (10000, 15)


In [2]:
# ── FEATURE ENGINEERING ────────────────────
import numpy as np

# Flag 1: Low temp differential (HDF danger zone)
df['low_temp_diff_flag'] = np.where(
    df['temp_diff'] < 8.6, 1, 0)

# Flag 2: High torque (above 75th percentile)
torque_threshold = df['Torque [Nm]'].quantile(0.75)
df['high_torque_flag'] = np.where(
    df['Torque [Nm]'] > torque_threshold, 1, 0)

# Flag 3: High tool wear (above 75th percentile)
wear_threshold = df['Tool wear [min]'].quantile(0.75)
df['high_tool_wear_flag'] = np.where(
    df['Tool wear [min]'] > wear_threshold, 1, 0)

# Verify — check failure rate inside vs outside each flag
for flag in ['low_temp_diff_flag',
              'high_torque_flag',
              'high_tool_wear_flag']:
    print(f"── {flag} ──")
    print(df.groupby(flag)['Machine failure']
          .agg(['count', 'mean']).round(4))
    print()

── low_temp_diff_flag ──
                    count   mean
low_temp_diff_flag              
0                    9280  0.023
1                     720  0.175

── high_torque_flag ──
                  count    mean
high_torque_flag               
0                  7529  0.0126
1                  2471  0.0987

── high_tool_wear_flag ──
                     count    mean
high_tool_wear_flag               
0                     7504  0.0221
1                     2496  0.0693

